# BarClean irrelevant-feature test with learned cutpoints

Goal: test each candidate one at a time using the current six demos and the current `map_balanced_pooled` configuration. All four cutpoints are learned; no true cutpoint or true τ is passed to the learner.

Acceptance: the candidate is inactive in S1–S5 and does not materially displace the learned cuts relative to the seven-feature baseline.

In [1]:
from dataclasses import replace
import json
from pathlib import Path
import time

import numpy as np

from envs.BarClean import load_BarClean
from experiments.config_loader import deep_merge
from experiments.unified_experiment import _load_method_config
from methods.wrappers.joint_map import _fit_single_map

ROOT = Path("/home/baiyu/LearnStageConstraints")
dataset_cfg = json.loads((ROOT / "configs/envs/BarClean.json").read_text())
override = dataset_cfg["method_overrides"]["map_balanced_pooled"]
bundle = load_BarClean(
    n_demos=dataset_cfg["n_demos"],
    seed=dataset_cfg["seed"],
    processed_demo_path=dataset_cfg["processed_demo_path"],
    source_demo_ids=dataset_cfg["source_demo_ids"],
)
env = bundle.env
base_schema = [dict(item) for item in env.get_feature_schema()]
base_selected = list(override["selected_raw_feature_ids"])

candidate_names = [
    "base_radial_dist",
    "base_azimuth",
    "bar_center_dist_3d",
    "nearest_bar_end_dist",
    "obstacle_dist_3d",
    "obstacle_in_tool_y",
    "bar_north_end_in_tool_y",
]
augmented_features = []
for trajectory, features, scene in zip(bundle.demos, bundle.features, env.demo_scenes):
    rotations = env._quat_to_matrix(trajectory[:, 3:7])
    tcp = trajectory[:, :3] + np.einsum("tij,j->ti", rotations, env.tcp_offset_local)
    tool_y = rotations[:, :, 1]
    bar_center, bar_axis, _ = env._bar_geometry_trace(trajectory, scene=scene)
    obstacle_center = env._obstacle_center_trace(trajectory, scene=scene)
    bar_north_end = bar_center + float(env.bar_outline_u[0]) * bar_axis
    bar_south_end = bar_center + float(env.bar_outline_u[1]) * bar_axis
    base_azimuth = np.arctan2(tcp[:, 1], tcp[:, 0])
    candidate_matrix = np.column_stack(
        [
            np.linalg.norm(tcp[:, :2], axis=1),
            base_azimuth,
            np.linalg.norm(tcp - bar_center, axis=1),
            np.minimum(
                np.linalg.norm(tcp - bar_north_end, axis=1),
                np.linalg.norm(tcp - bar_south_end, axis=1),
            ),
            np.linalg.norm(tcp - obstacle_center, axis=1),
            np.einsum("ti,ti->t", obstacle_center - tcp, tool_y),
            np.einsum("ti,ti->t", bar_north_end - tcp, tool_y),
        ]
    )
    augmented_features.append(np.column_stack([features, candidate_matrix]))

augmented_schema = base_schema + [
    {
        "id": len(base_schema) + index,
        "column_idx": len(base_schema) + index,
        "name": name,
        "unit": "rad" if name == "base_azimuth" else "m",
        "description": "irrelevant-feature learned-cutpoint diagnostic",
    }
    for index, name in enumerate(candidate_names)
]
env.get_feature_schema = lambda: [dict(item) for item in augmented_schema]
env.feature_schema = [dict(item) for item in augmented_schema]
diagnostic_bundle = replace(
    bundle,
    features=augmented_features,
    feature_schema=augmented_schema,
    true_taus=None,
    true_cutpoints=None,
)

effective_cfg = deep_merge(
    _load_method_config("map_balanced_pooled"),
    override,
)
effective_cfg.update(
    {
        "fixed_true_cutpoint_prefix": 0,
        "fixed_true_cutpoint_indices": [],
        "disable_plots": True,
        "save_paper_figures": False,
        "verbose": False,
        "map_demo_num_workers": 1,
    }
)
print(
    {
        "source_demo_ids": dataset_cfg["source_demo_ids"],
        "demo_lengths": [len(demo) for demo in bundle.demos],
        "base_features": base_selected,
        "candidates": candidate_names,
        "true_cutpoints_passed": diagnostic_bundle.true_cutpoints,
        "max_iter": effective_cfg["max_iter"],
        "method_seed": effective_cfg["seed"],
        "map_eq_sigma_mode": effective_cfg["map_eq_sigma_mode"],
    }
)

{'source_demo_ids': [3, 5, 6, 7, 8, 9], 'demo_lengths': [127, 143, 156, 116, 118, 123], 'base_features': ['obs_dist', 'table_dist', 'lateral_offset', 'axial_offset', 'tool_pitch', 'tool_roll', 'tool_yaw'], 'candidates': ['base_radial_dist', 'base_azimuth', 'bar_center_dist_3d', 'nearest_bar_end_dist', 'obstacle_dist_3d', 'obstacle_in_tool_y', 'bar_north_end_in_tool_y'], 'true_cutpoints_passed': None, 'max_iter': 8, 'method_seed': 0, 'map_eq_sigma_mode': 'profiled_upper_bound'}


In [2]:
MODE_LABELS = {0.0: "inactive", 1.0: "eq", -1.0: "lb", 2.0: "ub"}

def fit_candidate(candidate_name=None):
    selected = base_selected + ([] if candidate_name is None else [candidate_name])
    kwargs = dict(effective_cfg)
    kwargs["selected_raw_feature_ids"] = selected
    feature_count = len(selected)
    kwargs["map_activation_prior"] = [0.5] * feature_count
    kwargs["map_active_mode_prior"] = {
        mode: [1.0 / 3.0] * feature_count
        for mode in ("eq", "lb", "ub")
    }
    started = time.perf_counter()
    result = _fit_single_map(kwargs, diagnostic_bundle)
    elapsed = time.perf_counter() - started
    model = result["model"]
    signature = np.asarray(model.shared_activation_signature_mean, dtype=float)
    modes = {
        name: [
            MODE_LABELS[float(value)]
            for value in signature[:, model.feature_name_to_local_idx[name]]
        ]
        for name in selected
    }
    return {
        "candidate": candidate_name or "baseline",
        "elapsed_s": elapsed,
        "cutpoints": np.asarray(result["cutpoints_hat"], dtype=int),
        "modes": modes,
        "loss": float(result["total_cost"]),
        "progress_cost": float(result["progress_cost"]),
        "constraint_cost": float(result["constraint_cost"]),
        "iterations": len(model.loss_total),
    }

results = {}

In [3]:
results["baseline"] = fit_candidate()
print("baseline elapsed_s", round(results["baseline"]["elapsed_s"], 2))
print("baseline iterations", results["baseline"]["iterations"])
print("baseline learned cutpoints")
for source_id, cuts in zip(dataset_cfg["source_demo_ids"], results["baseline"]["cutpoints"]):
    print(source_id, cuts.tolist())
print("baseline modes")
for name in base_selected:
    print(name, results["baseline"]["modes"][name])

baseline elapsed_s 456.02
baseline iterations 4
baseline learned cutpoints
3 [33, 62, 83, 109]
5 [35, 69, 94, 115]
6 [35, 69, 115, 138]
7 [26, 53, 82, 104]
8 [28, 56, 84, 103]
9 [27, 56, 80, 106]
baseline modes
obs_dist ['lb', 'inactive', 'inactive', 'eq', 'inactive']
table_dist ['inactive', 'eq', 'inactive', 'eq', 'inactive']
lateral_offset ['inactive', 'eq', 'inactive', 'inactive', 'inactive']
axial_offset ['inactive', 'inactive', 'inactive', 'eq', 'inactive']
tool_pitch ['inactive', 'eq', 'inactive', 'eq', 'inactive']
tool_roll ['inactive', 'eq', 'inactive', 'eq', 'inactive']
tool_yaw ['inactive', 'eq', 'inactive', 'eq', 'inactive']


In [4]:
learned_cut_bundle = replace(
    diagnostic_bundle,
    true_cutpoints=[
        np.asarray(cuts, dtype=int)
        for cuts in results["baseline"]["cutpoints"]
    ],
)

def fit_candidate_on_learned_cuts(candidate_name):
    selected = base_selected + [candidate_name]
    kwargs = dict(effective_cfg)
    kwargs.update(
        {
            "selected_raw_feature_ids": selected,
            "fixed_true_cutpoint_prefix": 0,
            "fixed_true_cutpoint_indices": [0, 1, 2, 3],
            "max_iter": 3,
            "map_activation_prior": [0.5] * len(selected),
            "map_active_mode_prior": {
                mode: [1.0 / 3.0] * len(selected)
                for mode in ("eq", "lb", "ub")
            },
        }
    )
    started = time.perf_counter()
    result = _fit_single_map(kwargs, learned_cut_bundle)
    model = result["model"]
    signature = np.asarray(model.shared_activation_signature_mean, dtype=float)
    local_index = model.feature_name_to_local_idx[candidate_name]
    return {
        "candidate": candidate_name,
        "elapsed_s": time.perf_counter() - started,
        "modes": [
            MODE_LABELS[float(value)]
            for value in signature[:, local_index]
        ],
        "cutpoints": np.asarray(result["cutpoints_hat"], dtype=int),
        "iterations": len(model.loss_total),
    }

In [5]:
learned_cut_results = {}
for candidate_name in candidate_names:
    learned_cut_results[candidate_name] = fit_candidate_on_learned_cuts(candidate_name)
    item = learned_cut_results[candidate_name]
    assert np.array_equal(item["cutpoints"], results["baseline"]["cutpoints"])
    print(
        candidate_name,
        "modes=", item["modes"],
        "all_inactive=", all(mode == "inactive" for mode in item["modes"]),
        "elapsed_s=", round(item["elapsed_s"], 2),
        flush=True,
    )

base_radial_dist modes= ['inactive', 'eq', 'inactive', 'inactive', 'inactive'] all_inactive= False elapsed_s= 3.31
base_azimuth modes= ['inactive', 'inactive', 'ub', 'eq', 'inactive'] all_inactive= False elapsed_s= 3.28
bar_center_dist_3d modes= ['inactive', 'inactive', 'inactive', 'eq', 'inactive'] all_inactive= False elapsed_s= 3.3
nearest_bar_end_dist modes= ['inactive', 'lb', 'inactive', 'lb', 'inactive'] all_inactive= False elapsed_s= 3.62
obstacle_dist_3d modes= ['inactive', 'inactive', 'inactive', 'eq', 'inactive'] all_inactive= False elapsed_s= 3.72
obstacle_in_tool_y modes= ['inactive', 'inactive', 'inactive', 'inactive', 'inactive'] all_inactive= True elapsed_s= 3.88
bar_north_end_in_tool_y modes= ['inactive', 'inactive', 'inactive', 'inactive', 'inactive'] all_inactive= True elapsed_s= 3.57


In [6]:
learned_cut_summary = [
    {
        "feature": name,
        "modes": " ".join(
            f"S{stage + 1}:{mode}"
            for stage, mode in enumerate(learned_cut_results[name]["modes"])
        ),
        "all_inactive": all(
            mode == "inactive"
            for mode in learned_cut_results[name]["modes"]
        ),
    }
    for name in candidate_names
]
learned_cut_summary

[{'feature': 'base_radial_dist',
  'modes': 'S1:inactive S2:eq S3:inactive S4:inactive S5:inactive',
  'all_inactive': False},
 {'feature': 'base_azimuth',
  'modes': 'S1:inactive S2:inactive S3:ub S4:eq S5:inactive',
  'all_inactive': False},
 {'feature': 'bar_center_dist_3d',
  'modes': 'S1:inactive S2:inactive S3:inactive S4:eq S5:inactive',
  'all_inactive': False},
 {'feature': 'nearest_bar_end_dist',
  'modes': 'S1:inactive S2:lb S3:inactive S4:lb S5:inactive',
  'all_inactive': False},
 {'feature': 'obstacle_dist_3d',
  'modes': 'S1:inactive S2:inactive S3:inactive S4:eq S5:inactive',
  'all_inactive': False},
 {'feature': 'obstacle_in_tool_y',
  'modes': 'S1:inactive S2:inactive S3:inactive S4:inactive S5:inactive',
  'all_inactive': True},
 {'feature': 'bar_north_end_in_tool_y',
  'modes': 'S1:inactive S2:inactive S3:inactive S4:inactive S5:inactive',
  'all_inactive': True}]

In [7]:
quoted_names = [
    "bar_task_origin_dist_3d",
    "nearest_bar_endpoint_reference_dist",
]
quoted_features = []
for trajectory, features, scene in zip(bundle.demos, augmented_features, env.demo_scenes):
    rotations = env._quat_to_matrix(trajectory[:, 3:7])
    tcp = trajectory[:, :3] + np.einsum("tij,j->ti", rotations, env.tcp_offset_local)
    bar_center, bar_axis, _ = env._bar_geometry_trace(trajectory, scene=scene)
    task_origin = bar_center - np.outer(
        (bar_center - env.table_surface_point[None, :]) @ env.table_normal,
        env.table_normal,
    )
    north_reference = task_origin + float(env.bar_outline_u[0]) * bar_axis
    south_reference = task_origin + float(env.bar_outline_u[1]) * bar_axis
    quoted_features.append(
        np.column_stack(
            [
                features,
                np.linalg.norm(tcp - task_origin, axis=1),
                np.minimum(
                    np.linalg.norm(tcp - north_reference, axis=1),
                    np.linalg.norm(tcp - south_reference, axis=1),
                ),
            ]
        )
    )

quoted_schema = augmented_schema + [
    {
        "id": len(augmented_schema) + index,
        "column_idx": len(augmented_schema) + index,
        "name": name,
        "unit": "m",
        "description": "exact feature definition used by the earlier true-cutpoint screen",
    }
    for index, name in enumerate(quoted_names)
]
env.get_feature_schema = lambda: [dict(item) for item in quoted_schema]
env.feature_schema = [dict(item) for item in quoted_schema]
quoted_bundle = replace(
    learned_cut_bundle,
    features=quoted_features,
    feature_schema=quoted_schema,
)

def fit_quoted_feature(name):
    selected = base_selected + [name]
    kwargs = dict(effective_cfg)
    kwargs.update(
        {
            "selected_raw_feature_ids": selected,
            "fixed_true_cutpoint_indices": [0, 1, 2, 3],
            "max_iter": 3,
            "map_activation_prior": [0.5] * len(selected),
            "map_active_mode_prior": {
                mode: [1.0 / 3.0] * len(selected)
                for mode in ("eq", "lb", "ub")
            },
        }
    )
    result = _fit_single_map(kwargs, quoted_bundle)
    model = result["model"]
    signature = np.asarray(model.shared_activation_signature_mean, dtype=float)
    index = model.feature_name_to_local_idx[name]
    return [MODE_LABELS[float(value)] for value in signature[:, index]]

quoted_results = {name: fit_quoted_feature(name) for name in quoted_names}
quoted_results

{'bar_task_origin_dist_3d': ['inactive',
  'inactive',
  'inactive',
  'eq',
  'inactive'],
 'nearest_bar_endpoint_reference_dist': ['inactive',
  'lb',
  'inactive',
  'eq',
  'inactive']}